# Notebook 02 — Architecture Comparison: FNO vs F-FNO

This is the main replication experiment. We train both models on the same
data, with the same optimizer and schedule, and compare:

1. **Parameter count** — should show F-FNO uses ~10x fewer spectral parameters
2. **Test N-MSE** — paper reports F-FNO is ~35% better than FNO++ at 24 layers
3. **Scaling with depth** — paper Fig. 3 shows FNO gets worse with depth,
   F-FNO keeps improving. We test this at {4, 8, 12} layers (trimmed from
   the paper's {4, 8, 12, 16, 20, 24} to fit the time budget)
4. **Rollout stability** — how errors accumulate over autoregressive prediction

**Note on scale:** Paper trains ~100k steps; we train ~2000 steps. Absolute
numbers won't match the paper, but the relative trends should.

In [ ]:
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import torch
import numpy as np
import matplotlib.pyplot as plt

from train import train, TrainConfig
from models import FNO2d, FFNO2d

device = 'cuda' if torch.cuda.is_available() else 'cpu'
DATA_DIR = PROJECT_ROOT / 'ns_data'
RUNS_DIR = PROJECT_ROOT / 'runs'
RESULTS_DIR = PROJECT_ROOT / 'results'
RESULTS_DIR.mkdir(exist_ok=True)

## Parameter-count comparison (no training needed)

First, confirm the theoretical claim about parameter counts. Paper claim:

$$\text{FNO:  } O(LH^2 M^D) \qquad \text{F-FNO:  } O(LH^2 M D)$$

For $D=2$, $H=32$, $M=12$, the ratio should be $M/D = 6$ for the spectral
layers alone; overall model should also be much smaller for F-FNO.

In [ ]:
import pandas as pd

rows = []
for n_layers in [4, 8, 12, 16, 24]:
    fno = FNO2d(n_layers=n_layers, hidden=32, modes1=12, modes2=12)
    ffno = FFNO2d(n_layers=n_layers, hidden=32, modes_x=12, modes_y=12)
    rows.append({
        'layers': n_layers,
        'FNO params': fno.count_parameters(),
        'F-FNO params': ffno.count_parameters(),
        'ratio': fno.count_parameters() / ffno.count_parameters(),
    })
    del fno, ffno

df_params = pd.DataFrame(rows)
print(df_params.to_string(index=False))
df_params.to_csv(RESULTS_DIR / 'parameter_counts.csv', index=False)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(df_params['layers'], df_params['FNO params'] / 1e6, 'o-', label='FNO', lw=2)
ax.plot(df_params['layers'], df_params['F-FNO params'] / 1e6, 's-', label='F-FNO', lw=2)
ax.set_xlabel('Number of layers')
ax.set_ylabel('Parameters (millions)')
ax.set_title('Parameter count vs depth (H=32, M=12)')
ax.legend()
ax.grid(alpha=0.3)
ax.set_yscale('log')
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'param_count.png', dpi=120)
plt.show()

## Train at varying depth

Train FNO and F-FNO at layer counts {4, 8, 12}. We use the same optimizer
settings for both (AdamW, cosine decay from 2.5e-3). Input noise and
normalization are both on — these are the standard F-FNO tricks and we
give FNO the same advantage so the comparison isolates the architectural
difference.

In [ ]:
# Master switch: depth sweep
DEPTHS = [4, 8, 12]
MODELS = ['fno', 'ffno']

# Conservative settings to fit in a ~2 hour Colab session
# (6 runs x ~15-20 min each = ~90-120 min total)
N_EPOCHS = 20
BATCH_SIZE = 20

all_results = {}
for model_type in MODELS:
    for n_layers in DEPTHS:
        key = f'{model_type}_L{n_layers}'
        print(f'\n{"="*60}\nTraining {key}\n{"="*60}')
        cfg = TrainConfig(
            model_type=model_type,
            hidden=32,
            modes=12,
            n_layers=n_layers,
            data_dir=str(DATA_DIR),
            batch_size=BATCH_SIZE,
            n_epochs=N_EPOCHS,
            lr=2.5e-3,
            weight_decay=1e-4,
            warmup_steps=500,
            use_cosine_decay=True,
            input_noise_std=1e-3,
            device=device,
            seed=0,
            out_dir=str(RUNS_DIR / key),
            # If training takes too long, stop at 18 minutes per run
            max_train_seconds=18 * 60,
        )
        result = train(cfg)
        all_results[key] = result
        torch.save(all_results, RESULTS_DIR / 'depth_sweep_results.pt')

## Results: final test N-MSE vs depth

Main plot replicating paper Fig. 3 in spirit: does F-FNO scale better with
depth than FNO? Expected pattern — FNO error flat or increasing, F-FNO
error decreasing.

In [ ]:
# If you restarted the notebook, load previous results
if not all_results:
    all_results = torch.load(RESULTS_DIR / 'depth_sweep_results.pt', weights_only=False)

rows = []
for key, r in all_results.items():
    model_type, layer_tag = key.split('_')
    rows.append({
        'model': model_type,
        'layers': int(layer_tag.lstrip('L')),
        'test_NMSE': r['test_loss'],
        'best_val_NMSE': r['best_val_loss'],
        'params': r['n_params'],
        'time_sec': r['total_time_sec'],
    })
df = pd.DataFrame(rows).sort_values(['model', 'layers'])
print(df.to_string(index=False))
df.to_csv(RESULTS_DIR / 'depth_sweep_summary.csv', index=False)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# Left: test N-MSE vs depth
for model_type, color in [('fno', 'C0'), ('ffno', 'C1')]:
    sub = df[df.model == model_type].sort_values('layers')
    label = 'FNO' if model_type == 'fno' else 'F-FNO'
    axes[0].plot(sub.layers, sub.test_NMSE * 100, 'o-', color=color, lw=2, label=label, markersize=8)
axes[0].set_xlabel('Number of layers')
axes[0].set_ylabel('Test N-MSE (%)')
axes[0].set_title('Accuracy vs depth')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Right: test N-MSE vs number of params (a fairer efficiency view)
for model_type, color in [('fno', 'C0'), ('ffno', 'C1')]:
    sub = df[df.model == model_type].sort_values('params')
    label = 'FNO' if model_type == 'fno' else 'F-FNO'
    axes[1].plot(sub.params / 1e6, sub.test_NMSE * 100, 'o-', color=color, lw=2, label=label, markersize=8)
    for _, row in sub.iterrows():
        axes[1].annotate(f"L={int(row.layers)}", (row.params / 1e6, row.test_NMSE * 100),
                         textcoords='offset points', xytext=(6, 4), fontsize=9)
axes[1].set_xlabel('Parameters (millions)')
axes[1].set_ylabel('Test N-MSE (%)')
axes[1].set_title('Parameter efficiency')
axes[1].set_xscale('log')
axes[1].legend()
axes[1].grid(alpha=0.3, which='both')

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'depth_comparison.png', dpi=120)
plt.show()

## Training curves

Shows whether training is stable for deeper networks. One of the paper's
key claims is that FNO training becomes unstable / fails to converge as
depth grows past ~16 layers, while F-FNO remains stable.

In [ ]:
fig, axes = plt.subplots(1, len(DEPTHS), figsize=(5 * len(DEPTHS), 4), sharey=True)
for ax, L in zip(axes, DEPTHS):
    for model_type, color in [('fno', 'C0'), ('ffno', 'C1')]:
        r = all_results[f'{model_type}_L{L}']['history']
        if 'val_loss' in r and len(r['val_loss']) > 0:
            val_steps = r['step'][-len(r['val_loss']):]
            ax.plot(val_steps, r['val_loss'], '-', color=color,
                    label=f"{'F-FNO' if model_type == 'ffno' else 'FNO'}")
    ax.set_title(f'{L} layers')
    ax.set_xlabel('Training step')
    ax.set_yscale('log')
    ax.grid(alpha=0.3, which='both')
    ax.legend()
axes[0].set_ylabel('Validation N-MSE')
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'training_curves.png', dpi=120)
plt.show()

## Rollout stability

Use the best models (8-layer FNO and F-FNO) to autoregressively predict
10 timesteps starting from $w_0$. Error should grow with time, but F-FNO's
error should grow more slowly if the learned operator is more accurate per
step.

In [ ]:
from data.dataset import RolloutDataset
from train import evaluate_rollout, build_model

L_pick = 8    # compare at 8 layers (sweet spot)
train_stats = torch.load(DATA_DIR / 'train.pt', weights_only=True)
stats = {'mean': train_stats.mean().item(), 'std': train_stats.std().item()}
del train_stats
rollout_ds = RolloutDataset(DATA_DIR / 'test.pt', normalize=True, stats=stats)

rollout_results = {}
for model_type in ['fno', 'ffno']:
    key = f'{model_type}_L{L_pick}'
    cfg = TrainConfig(**all_results[key]['config'])
    model = build_model(cfg).to(device)
    model.load_state_dict(torch.load(RUNS_DIR / key / 'best.pt', weights_only=True))
    rollout_results[model_type] = evaluate_rollout(model, rollout_ds, device=device)
    del model

In [ ]:
plt.figure(figsize=(8, 4.5))
for model_type, label, color in [('fno', 'FNO', 'C0'), ('ffno', 'F-FNO', 'C1')]:
    m = np.array(rollout_results[model_type]['per_step_mean']) * 100
    s = np.array(rollout_results[model_type]['per_step_std']) * 100
    steps = np.arange(1, len(m) + 1)
    plt.plot(steps, m, 'o-', color=color, label=label, lw=2, markersize=7)
    plt.fill_between(steps, m - s, m + s, color=color, alpha=0.2)
plt.xlabel('Rollout step')
plt.ylabel('N-MSE (%)')
plt.title(f'Autoregressive rollout error ({L_pick}-layer models)')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'rollout_error.png', dpi=120)
plt.show()

## Qualitative comparison

Side-by-side: ground truth vs FNO prediction vs F-FNO prediction at one
held-out test trajectory.

In [ ]:
# Pick a test trajectory, roll it out with both models, plot final step
test_data = torch.load(DATA_DIR / 'test.pt', weights_only=True).to(device)
traj_idx = 0
traj = test_data[traj_idx]                                          # (T, H, W)
coords = rollout_ds.coords.to(device)
mean, std = stats['mean'], stats['std']

rollout_preds = {}
for model_type in ['fno', 'ffno']:
    key = f'{model_type}_L{L_pick}'
    cfg = TrainConfig(**all_results[key]['config'])
    model = build_model(cfg).to(device)
    model.load_state_dict(torch.load(RUNS_DIR / key / 'best.pt', weights_only=True))
    model.eval()

    preds = [traj[0]]
    w = ((traj[0] - mean) / std).unsqueeze(0)
    with torch.no_grad():
        for t in range(1, traj.size(0)):
            x_in = torch.cat([w, coords.unsqueeze(0)], dim=1)
            w = model(x_in).squeeze(1)
            preds.append(w.squeeze(0) * std + mean)
    rollout_preds[model_type] = torch.stack(preds)
    del model

show_t = [1, 5, 10, 15, 19]
fig, axes = plt.subplots(3, len(show_t), figsize=(3.2 * len(show_t), 9))
vmax = traj.abs().max().item()
for col, t in enumerate(show_t):
    axes[0, col].imshow(traj[t].cpu(), cmap='RdBu_r', vmin=-vmax, vmax=vmax)
    axes[0, col].set_title(f't = {t * 0.1:.1f}s')
    axes[1, col].imshow(rollout_preds['fno'][t].cpu(), cmap='RdBu_r', vmin=-vmax, vmax=vmax)
    axes[2, col].imshow(rollout_preds['ffno'][t].cpu(), cmap='RdBu_r', vmin=-vmax, vmax=vmax)
    for row in range(3):
        axes[row, col].axis('off')
axes[0, 0].set_ylabel('Ground truth', fontsize=12)
axes[1, 0].set_ylabel('FNO', fontsize=12)
axes[2, 0].set_ylabel('F-FNO', fontsize=12)
for row, label in enumerate(['Ground truth', 'FNO (rollout)', 'F-FNO (rollout)']):
    axes[row, 0].text(-5, 32, label, fontsize=12, ha='right', va='center', rotation=90)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'qualitative_rollout.png', dpi=120, bbox_inches='tight')
plt.show()

## Summary

Look for the following patterns in the results above:

| Claim from paper | Expected in our results |
|---|---|
| F-FNO uses ~10× fewer params | ✓ visible in param table |
| F-FNO scales better with depth | F-FNO loss decreases with L; FNO loss plateaus or grows |
| F-FNO is more accurate at equal params | F-FNO curve below FNO in efficiency plot |
| F-FNO has smaller rollout error growth | F-FNO curve below FNO in rollout plot |

If your trends match, we have a successful replication. Absolute numbers
will differ from the paper due to the much smaller training budget.